# Import Requirements

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, BertTokenizer, AutoModel, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
from torch.nn.functional import softmax
import re
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import mean_absolute_error
from tqdm import tqdm
from scipy.stats import pearsonr

In [ ]:
pelajar_path = ""
kunci_path = ""
model_indobert_ft =""
model_indobert = ""

# Load Data

In [ ]:
df_pelajar = pd.read_csv(pelajar_path)
df_kunci = pd.read_csv(kunci_path)

In [ ]:
df_kunci

,kode_soal,kunci_jawaban,soal
0,1,Melakukan penyuluhan tentang perilaku bijak me...,Perhatikan informasi berikut!\nKemudahan akses...
1,2,Persatuan bangsa Indonesia belum mampu mengala...,Sebelum kebangkitan nasional perjuangan bangsa...
2,3,Semangat persatuan dan kesatuan bangsa Indones...,Jelaskan alasan mengapa Peristiwa Sumpah Pemud...
3,4,Memperbaiki kualitas pekerja Indonesia dengan ...,Berdasarkan data Badan Pusat Statistik BPS ten...


In [ ]:
# hapus kolom tidak relevan di df_kunci
df_kunci = df_kunci.drop(columns=['soal'])

# Preprocessing

In [ ]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s-]', '', text)
    text = re.sub(r'-', ' ', text)
    return text

list_dict = {
    "dg": "dengan",
    "dgn": "dengan",
    "yg": "yang",
    "dpt": "dapat",
    "dlm": "dalam",
    "sdh": "sudah",
    "blm": "belum",
    "tp": "tapi",
    "tpi": "tapi"
}

def normalize_word(text, dictionary):
    words = text.split()
    words = [dictionary[word] if word in dictionary else word for word in words]
    return ' '.join(words)


df_pelajar['jawaban_siswa'] = df_pelajar['jawaban_siswa'].apply(normalize_text)
df_pelajar['jawaban_siswa'] = df_pelajar['jawaban_siswa'].apply(lambda x: normalize_word(x, list_dict))
df_kunci['kunci_jawaban'] = df_kunci['kunci_jawaban'].apply(normalize_text)
df_kunci['kunci_jawaban'] = df_kunci['kunci_jawaban'].apply(lambda x: normalize_word(x, list_dict))

# Load Model

In [ ]:
# Model IndoBERT
tokenizer_indobert = BertTokenizer.from_pretrained(model_indobert)
model_indobert = AutoModel.from_pretrained(model_indobert)

# Model IndoBERT setelah Fine-Tune
model_ai = AutoModelForSequenceClassification.from_pretrained(model_indobert_ft)
tokenizer_ai = BertTokenizer.from_pretrained(model_indobert_ft)

# Penilaian

1. Hitung Similaritas

In [ ]:
# Embedding
def get_embedding(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding='max_length', max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)

    token_embeddings = outputs.last_hidden_state
    attention_mask = inputs['attention_mask']
    mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size())
    sum_embeddings = torch.sum(token_embeddings * mask_expanded, 1)
    sum_mask = mask_expanded.sum(1)
    embeddings = sum_embeddings / sum_mask
    return embeddings

def convert_manual_score(manual_score):
    return manual_score*25

# Similaritas jawaban
def get_similarity_score(text_students, text_ref, model, tokenizer, nilai_manual):
    emb_students = get_embedding(text_students, model, tokenizer)
    emb_ref = get_embedding(text_ref, model, tokenizer)
    sim = F.cosine_similarity(emb_students, emb_ref).item()
    manual_score = convert_manual_score(nilai_manual)

    return {
        'similarity_raw': sim,
        'manual_score': manual_score
    }

In [ ]:
df = pd.merge(df_pelajar, df_kunci, on='kode_soal', how='inner')
df.sample(3)

,kode_soal,siswa,jawaban_siswa,nilai_manual,kunci_jawaban
73,2,S19,kesimpulannya persatuan di indonesia masih ber...,4,persatuan bangsa indonesia belum mampu mengala...
59,4,S15,melatih para pekerja agar bisa menyaingi para ...,4,memperbaiki kualitas pekerja indonesia dengan ...
56,1,S15,mengurangimembatasi anak anak bermain hp atau ...,4,melakukan penyuluhan tentang perilaku bijak me...


In [ ]:
similarities = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    emb_siswa = get_embedding(row['jawaban_siswa'], model_indobert, tokenizer_indobert)
    emb_kunci = get_embedding(row['kunci_jawaban'], model_indobert, tokenizer_indobert)
    sim = F.cosine_similarity(emb_siswa, emb_kunci).item()
    similarities.append(sim)

100%|██████████| 80/80 [05:45<00:00,  4.32s/it]


2. Konversi Nilai

In [ ]:
# konversi manual score ke rentang 100
def convert_manual_score(manual_score):
    return manual_score*25

df['nilai_manual'] = df['nilai_manual'].apply(convert_manual_score)

In [ ]:
# konversi similarity dikalikan langsung 100
def convert_similarity_score(similarity_score):
  return similarity_score*100

df['similarity'] = similarities
df['similarity'] = df['similarity'].round(4)
df['similarity_score'] = df['similarity'].apply(convert_similarity_score)

df.sample(3)

,kode_soal,siswa,jawaban_siswa,nilai_manual,kunci_jawaban,similarity,similarity_score
5,2,S2,kesadaran untuk bersatu menjadi kunci utama da...,100,persatuan bangsa indonesia belum mampu mengala...,0.9017,90.17
45,2,S12,kebangkitan nasional itu sangat penting karena...,100,persatuan bangsa indonesia belum mampu mengala...,0.8223,82.23
62,3,S16,karena pada saat itu para pemuda bekerja sama ...,25,semangat persatuan dan kesatuan bangsa indones...,0.7530,75.30


In [ ]:
# Evaluasi MAE hasil konversi similarity dikalikan langsung 100
mae1 = mean_absolute_error(df['nilai_manual'], df['similarity_score'])
print(f'Mean Absolute Error (MAE): {mae1:.4f}')

Mean Absolute Error (MAE): 23.1418


In [ ]:
jumlah_sama = (df['similarity_score'] == df['nilai_manual']).sum()
print(f"Jumlah nilai similaritas langsung yang sama dengan nilai manual: {jumlah_sama}")

Jumlah nilai similaritas langsung yang sama dengan nilai manual: 0


In [ ]:
# konversi nilai similarity menggunakan min-max
# nilai min-max bersifat eksplorasi dari data aktual
def similarity_konversi(sim, min_sim=0.75, max_sim=0.85):
    return max(0, min(100 * ((sim - min_sim) / (max_sim - min_sim)), 100))

df['similarity_score_new'] = df['similarity'].apply(similarity_konversi)
df.sample(5)

,kode_soal,siswa,jawaban_siswa,nilai_manual,kunci_jawaban,similarity,similarity_score,similarity_score_new
48,1,S13,melakukan pendampingan dan monitoring terhadap...,100,melakukan penyuluhan tentang perilaku bijak me...,0.8883,88.83,100.0
24,1,S7,orang tua harus lebih mengawasi anaknya dan me...,100,melakukan penyuluhan tentang perilaku bijak me...,0.8560,85.60,100.0
50,3,S13,pengingat akan pentingnya persatuan dan kesatu...,75,semangat persatuan dan kesatuan bangsa indones...,0.8466,84.66,96.6
64,1,S17,melakukan penyuluhan tentang adanya prilaku bi...,100,melakukan penyuluhan tentang perilaku bijak me...,0.9030,90.30,100.0
76,1,S20,melarang anak dibawah umur untuk main hp dan m...,50,melakukan penyuluhan tentang perilaku bijak me...,0.7933,79.33,43.3


In [ ]:
# Evaluasi MAE hasil konversi similarity menggunakan pendekatan normalisasi min-max
mae2 = mean_absolute_error(df['nilai_manual'], df['similarity_score_new'])
print(f'Mean Absolute Error (MAE): {mae2:.4f}')

Mean Absolute Error (MAE): 21.6200


In [ ]:
jumlah_sama = (df['similarity_score_new'] == df['nilai_manual']).sum()
print(f"Jumlah nilai similaritas langsung yang sama dengan nilai manual: {jumlah_sama}")

Jumlah nilai similaritas langsung yang sama dengan nilai manual: 26


3. Deteksi AI

In [ ]:
# Deteksi AI
def predict_ai(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits.squeeze()
        probs = softmax(logits, dim=-1)
        ai_prob = probs[0].item()

    return ai_prob * 100

# terapkan penalti
def apply_ai_penalty(score, ai_prob):
    return max(0, score*(1 - ai_prob/100))

In [ ]:
ai_probabilitas = []
score_final = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    student_answer = row['jawaban_siswa']
    nilai_similarity_konfersi = row['similarity_score_new']
    nilai_manual = row['nilai_manual']

    ai_probs = predict_ai(student_answer, model_ai, tokenizer_ai)
    final = apply_ai_penalty(nilai_similarity_konfersi, ai_probs)

    ai_probabilitas.append(round(ai_probs, 2))
    score_final.append(round(final, 2))

df['ai_probabilitas'] = ai_probabilitas
df['score_final'] = score_final

100%|██████████| 80/80 [00:18<00:00,  4.32it/s]


In [ ]:
df_hasil_akhir = df.drop(columns=['kunci_jawaban', 'similarity_score'])

In [ ]:
df_hasil_akhir = df_hasil_akhir.rename(columns={
    'siswa': 'Siswa',
    'jawaban_siswa': 'Jawaban',
    'similarity': 'Similarity',
    'similarity_score_new': 'Similarity (Konversi)',
    'ai_probabilitas': 'Probabilitas AI',
    'score_final': 'Nilai Akhir',
    'nilai_manual': 'Nilai Manual'
})
df_hasil_akhir.sample(5)

,kode_soal,Siswa,Jawaban,Nilai Manual,kunci_jawaban,Similarity,Similarity (Konversi),Probabilitas AI,Nilai Akhir
27,4,S7,sebaiknya negara mencegah adanya tenaga kerja ...,50.0,memperbaiki kualitas pekerja indonesia dengan ...,0.8300,80.0,0.32,79.74
35,4,S9,memperbanyak lapangan pekerjaan untuk masyarak...,50.0,memperbaiki kualitas pekerja indonesia dengan ...,0.7508,0.8,0.46,0.80
0,1,S1,orang tua harus membatasi waktu bermain hp ana...,50.0,melakukan penyuluhan tentang perilaku bijak me...,0.8586,100.0,0.34,99.66
49,2,S13,pentingnya peran nasional bangsa indonesia dal...,75.0,persatuan bangsa indonesia belum mampu mengala...,0.8111,61.1,0.66,60.69
76,1,S20,melarang anak dibawah umur untuk main hp dan m...,50.0,melakukan penyuluhan tentang perilaku bijak me...,0.7933,43.3,0.41,43.12


In [ ]:
df_hasil_akhir['Similarity'] = df_hasil_akhir['Similarity'].astype(float)
df_hasil_akhir['Similarity (Konversi)'] = df_hasil_akhir['Similarity (Konversi)'].astype(float)
df_hasil_akhir['Probabilitas AI'] = df_hasil_akhir['Probabilitas AI'].astype(float)

df_hasil_sama = df_hasil_akhir[df_hasil_akhir['Similarity (Konversi)'] == df_hasil_akhir['Nilai Manual']]
df_hasil_akhir_ai_tinggi = df_hasil_sama.sort_values(by='Probabilitas AI', ascending=False).reset_index(drop=True)
df_hasil_akhir_ai_tinggi.head()

,kode_soal,Siswa,Jawaban,Nilai Manual,kunci_jawaban,Similarity,Similarity (Konversi),Probabilitas AI,Nilai Akhir
0,2,S19,kesimpulannya persatuan di indonesia masih ber...,100.0,persatuan bangsa indonesia belum mampu mengala...,0.8756,100.0,23.68,76.32
1,3,S18,karena para pemuda dari berbagai daerah yang b...,100.0,semangat persatuan dan kesatuan bangsa indones...,0.8789,100.0,4.37,95.63
2,4,S2,pemerintah bisa memperbaiki kualitas para peke...,100.0,memperbaiki kualitas pekerja indonesia dengan ...,0.9346,100.0,1.07,98.93
3,3,S8,sumpah pemuda yang mengikrarkan persatuan bang...,100.0,semangat persatuan dan kesatuan bangsa indones...,0.8570,100.0,0.91,99.09
4,2,S2,kesadaran untuk bersatu menjadi kunci utama da...,100.0,persatuan bangsa indonesia belum mampu mengala...,0.9017,100.0,0.84,99.16
